# alpha-mipt-rag — прогон на Google Colab (A100, sglang)

RAG поверх корпуса `alfabank.ru` → submission CSV, на GPU. Код тянется из GitHub.

**Профиль:** embedder `bge-m3` + reranker `bge-reranker-v2-m3` (cuda) + генерация через **sglang**,
поднятый в фоне как локальный OpenAI-совместимый сервер. Из ноутбука к нему ходим по HTTP
через `openai`-клиент.

Генератор по умолчанию — `Qwen/Qwen2.5-14B-Instruct-AWQ` (ungated, system-роль OK, ~9 ГБ в 4-bit,
надёжный старт на A100 80 ГБ).

**Перед стартом:** Runtime → Change runtime type → **A100 GPU**. Затем Runtime → Run all.
Ячейка зависимостей один раз сама перезапустит runtime — после этого снова Run all.

## 0. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvidia-smi

## 1. Параметры

In [ ]:
GIT_URL = "https://github.com/Vesoore/alpha_mipt_rag.git"
WORKDIR = "/content/alpha_mipt_rag"

GEN_MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"   # ungated, system-роль OK, ~9 ГБ в 4-bit
# GEN_MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"        # крупнее (~19 ГБ), выше качество
# GEN_MODEL = "mistralai/Mistral-Small-24B-Instruct-2501"  # ~24B bf16, Apache-2.0, мультиязычный
HF_TOKEN = ""                                  # нужен только для gated-моделей

# sglang server (поднимается в фоне внутри Colab)
SGLANG_HOST = "127.0.0.1"
SGLANG_PORT = 30000
SGLANG_MEM_FRACTION = 0.65    # доля 80 ГБ под веса+KV-кэш sglang; остальное — embedder+reranker+слак
SGLANG_CONTEXT_LEN = 8192     # cap контекста — режет KV-кэш
SGLANG_MAX_RUN = 32           # параллельные запросы у сервера
SGLANG_DTYPE = "float16"      # AWQ-веса хранятся в int4, активации fp16

# Клиентская сторона (батч-генерация по HTTP)
N_QUESTIONS = None   # None = все ~6977; число = быстрая проверка
BATCH = 256          # размер чанка вопросов между записями чекпоинта
CONCURRENCY = 32     # параллельные HTTP-запросы к sglang (должно быть ≤ SGLANG_MAX_RUN)

BASE_URL = f"http://{SGLANG_HOST}:{SGLANG_PORT}/v1"
print("repo:", GIT_URL, "| model:", GEN_MODEL, "| sglang:", BASE_URL)

## 2. Клонировать код

In [ ]:
import os, shutil, subprocess, sys

os.chdir("/content")   # не оставаться внутри WORKDIR, иначе rmtree удалит CWD -> git clone exit 128
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.run(["git", "clone", "--depth", "1", GIT_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

for fn in ["websites.csv", "questions.csv", "sample_submission.csv"]:
    p = os.path.join("data", fn)
    assert os.path.exists(p), f"НЕТ {p} — данные должны быть в репозитории"
    print(f"  data/{fn}: {os.path.getsize(p)/1024/1024:.1f} MB")
print("WORKDIR:", os.getcwd())

## 3. Зависимости

Образ Colab ставит слишком свежий `torch 2.11+cu130`, под который у sglang нет CUDA-кернелов.
Поэтому здесь мы **сносим cu130-стек и ставим согласованный torch 2.6.0 / cu124** + sglang
(с flashinfer/sgl-kernel) + openai-клиент + retrieval-стек. Драйвер A100 (CUDA 13) cu124-колёса
запускает нормально.

Ячейка **один раз** перезапустит runtime; после этого снова Runtime → Run all.

In [ ]:
import subprocess, sys, os
MARK = "/content/.deps_done"

# JAX в этом ноутбуке не нужен (всё на torch), а bm25s при импорте безусловно прогревает
# jax.lax.top_k на GPU -> конфликт cuDNN с torch-cu124 -> XlaRuntimeError. Гасим JAX и
# просим transformers не грузить flax/tf (env — вторичная страховка, главный фикс — снос jax ниже).
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TF"] = "0"

def pip(*args, check=True):
    subprocess.run([sys.executable, "-m", "pip", *args], check=check)

# Образ Colab теперь ставит torch 2.11+cu130 — это СЛИШКОМ свежо для sglang:
#  * torchvision под cu130 не регистрирует нативные операторы ("torchvision::nms does not exist");
#  * sgl-kernel/flashinfer из sglang 0.4.6 вообще не имеют колёс под torch 2.11 / cu130.
# Поэтому сносим cu130-стек и ставим согласованный CUDA-12.4 torch, который sglang
# поддерживает. Драйвер A100 (CUDA 13.0) запускает cu124-колёса — драйвер обратно совместим.
TORCH_VER, TV_VER, TA_VER = "2.6.0", "0.21.0", "2.6.0"
CU = "cu124"
TORCH_INDEX = f"https://download.pytorch.org/whl/{CU}"
FLASHINFER_INDEX = f"https://flashinfer.ai/whl/{CU}/torch2.6"

pip("install", "-q", "--upgrade", "pip")

# 1) Сносим предустановленный cu130-стек, чтобы он не "удовлетворял" зависимости.
pip("uninstall", "-y", "-q", "torch", "torchvision", "torchaudio", "torchcodec", check=False)

# 2) Целостная тройка torch под cu124 (нативные операторы torchvision совпадают с torch).
pip("install", "-q", "--no-cache-dir",
    f"torch=={TORCH_VER}", f"torchvision=={TV_VER}", f"torchaudio=={TA_VER}",
    "--index-url", TORCH_INDEX)

# 3) sglang + его CUDA-кернелы. flashinfer берём из его wheel-индекса под torch2.6/cu124,
#    torch-тройку фиксируем в той же команде, чтобы резолвер её не трогал.
#    compressed-tensors пиним <0.10: свежий (0.10+) импортирует transformers.masking_utils,
#    которого нет у transformers из sglang 0.4.6 -> ModuleNotFoundError при старте сервера.
#    Мы compressed-tensors не используем (модель AWQ/Marlin) — нужен лишь чистый импорт.
pip("install", "-q",
    "sglang[all]>=0.4.6.post1,<0.4.7",
    "compressed-tensors<0.10",
    f"torch=={TORCH_VER}", f"torchvision=={TV_VER}", f"torchaudio=={TA_VER}",
    "--find-links", FLASHINFER_INDEX)

# 4) retrieval/eval-стек + HTTP-клиент (torch не трогают — 2.6.0 их устраивает).
pip("install", "-q",
    "openai>=1.50",
    "sentence-transformers>=3.0", "faiss-cpu",
    "polars>=1.0", "pydantic>=2.0", "pyyaml", "tqdm", "bm25s>=0.2",
    "requests")

# Пакеты, ломающие импорт на этом стеке (сносим в самом конце, после всех install):
#  - torchcodec: нужен FFmpeg, несовместим с torch.
#  - kernels (HF kernel-hub): свежий transformers создаёт LayerRepository без version,
#    а новый kernels требует version/revision -> ValueError на импорте transformers.
#  - jax/jaxlib (+cuda-плагины): bm25s при импорте прогревает jax.lax.top_k на GPU ->
#    конфликт cuDNN с torch -> XlaRuntimeError. Без jax bm25s считает top-k на numpy.
pip("uninstall", "-y", "-q",
    "torchcodec", "kernels",
    "jax", "jaxlib", "jax-cuda12-plugin", "jax-cuda12-pjrt", check=False)
os.environ["DISABLE_KERNEL_MAPPING"] = "1"

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
if not os.path.exists(MARK):
    open(MARK, "w").close()
    print("зависимости установлены — перезапускаю runtime, после рестарта: Runtime → Run all")
    os.kill(os.getpid(), 9)
else:
    import torchvision, sglang, openai
    print("OK: torchvision", torchvision.__version__,
          "| sglang", sglang.__version__, "| openai", openai.__version__)

In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA недоступна — runtime должен быть A100 GPU"
print(torch.cuda.get_device_name(0))

## 4. HF-токен (только для gated-моделей)

In [ ]:
import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HF login OK")
else:
    print("HF_TOKEN пуст — ок для ungated", GEN_MODEL)

## 5. config.yaml → профиль openai_api (sglang)

Чиним id генератора, переключаем backend на `openai_api`, прописываем base_url и concurrency.

In [ ]:
import yaml
with open("config.yaml", encoding="utf-8") as f:
    c = yaml.safe_load(f)
c["embedder"]["device"] = "cuda"
c["embedder"]["batch_size"] = 8            # маленький batch -> нет пика памяти на длинных чанках
c["reranker"]["device"] = "cuda"
c["reranker"]["batch_size"] = 16

c["generator"]["backend"] = "openai_api"
c["generator"]["model"] = GEN_MODEL
c["generator"]["base_url"] = BASE_URL
c["generator"]["api_key"] = "EMPTY"
c["generator"]["max_concurrency"] = CONCURRENCY
c["generator"]["request_timeout"] = 300.0
# vLLM-поля больше не используются — оставляем как заглушки
c["generator"].pop("quantization", None)
c["generator"].pop("gpu_memory_utilization", None)
c["generator"].pop("max_model_len", None)
c["generator"].pop("max_num_seqs", None)
c["generator"].pop("enforce_eager", None)
c["generator"].pop("tensor_parallel_size", None)

with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(c, f, allow_unicode=True, sort_keys=False)
print("generator:", c["generator"]["model"], "→", c["generator"]["base_url"])
print("embedder:", c["embedder"]["model"])

## 6. Индекс (chunks + FAISS + BM25)

Артефакты не в git → строим на месте (~3-5 мин). После этого ОБЯЗАТЕЛЬНО возвращаем
кэш PyTorch ОС — иначе sglang-сервер не получит свою долю памяти.

In [ ]:
import os
# JAX не должен трогать GPU (иначе конфликт cuDNN с torch -> XlaRuntimeError);
# transformers не должен грузить flax/tf. Дублируем на случай чистого запуска ячейки.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TF"] = "0"

import gc, torch
from rag.config import load_config, seed_everything
cfg = load_config()
seed_everything(cfg.seed)

needed = [cfg.resolve(cfg.paths.chunks_parquet),
          cfg.resolve(cfg.paths.faiss_index),
          cfg.resolve(cfg.paths.bm25_pickle)]
if [p for p in needed if not p.exists()]:
    print("строю индекс…")
    from rag.data import load_and_clean
    from rag.chunking import chunk_dataframe, save_chunks
    from rag.retrieval.dense import DenseRetriever
    from rag.retrieval.bm25 import BM25Retriever
    docs = load_and_clean(cfg.resolve(cfg.paths.websites_csv), cfg.cleaning)
    chunks_df = chunk_dataframe(docs, cfg.chunking)
    save_chunks(chunks_df, cfg.resolve(cfg.paths.chunks_parquet))
    db = DenseRetriever(cfg.embedder); db.build_index(chunks_df); db.save(cfg.resolve(cfg.paths.faiss_index))
    bb = BM25Retriever(); bb.build_index(chunks_df); bb.save(cfg.resolve(cfg.paths.bm25_pickle))
    del db, bb
    print("индекс готов")
else:
    print("индекс уже на месте")

# КЛЮЧЕВОЕ: вернуть драйверу кэш PyTorch от эмбеддинга, иначе ~66 ГБ висят и sglang не стартует.
gc.collect()
torch.cuda.empty_cache()
print("после очистки занято ядром:", round(torch.cuda.memory_reserved()/1e9, 1), "ГБ;",
      "свободно на GPU:", round(torch.cuda.mem_get_info()[0]/1e9, 1), "ГБ")

In [ ]:
!nvidia-smi

## 7. Запустить sglang-сервер (в фоне)

Поднимаем `python -m sglang.launch_server` как отдельный процесс, ждём готовности по
`/health`. Логи пишем в `/content/sglang.log` — если что-то пошло не так, `!tail` его.

`--mem-fraction-static` — доля **всей** видеопамяти, которую sglang забронирует под
веса+KV-кэш. На A100 80 ГБ при 0.65 это ~52 ГБ; embedder+reranker и временные буферы
помещаются в оставшиеся ~28 ГБ.

In [ ]:
import os, sys, time, signal, subprocess, requests

LOG_PATH = "/content/sglang.log"
log_f = open(LOG_PATH, "w")

cmd = [
    sys.executable, "-m", "sglang.launch_server",
    "--model-path", GEN_MODEL,
    "--host", SGLANG_HOST,
    "--port", str(SGLANG_PORT),
    "--mem-fraction-static", str(SGLANG_MEM_FRACTION),
    "--context-length", str(SGLANG_CONTEXT_LEN),
    "--max-running-requests", str(SGLANG_MAX_RUN),
    "--dtype", SGLANG_DTYPE,
    "--disable-cuda-graph",  # стабильнее старт, меньше памяти
]
# AWQ-модели грузятся быстрее ядром Marlin
if "AWQ" in GEN_MODEL.upper():
    cmd += ["--quantization", "awq_marlin"]

env = os.environ.copy()
env.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

sglang_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
print("sglang PID:", sglang_proc.pid, "→ log:", LOG_PATH)

# Ждём готовности (модель ~9 ГБ, скачка+прогрев до ~6 мин на свежем кэше)
HEALTH = f"http://{SGLANG_HOST}:{SGLANG_PORT}/health"
MODELS = f"http://{SGLANG_HOST}:{SGLANG_PORT}/v1/models"
t0 = time.time()
TIMEOUT = 900    # 15 минут с запасом
while True:
    if sglang_proc.poll() is not None:
        log_f.flush()
        raise RuntimeError(f"sglang упал (exit={sglang_proc.returncode}); см. {LOG_PATH}")
    try:
        r = requests.get(HEALTH, timeout=3)
        if r.status_code == 200:
            # ждём ещё пока модель реально отвечает
            m = requests.get(MODELS, timeout=5)
            if m.status_code == 200 and m.json().get("data"):
                print(f"sglang готов за {time.time()-t0:.0f}s; модели:",
                      [x['id'] for x in m.json()['data']])
                break
    except requests.RequestException:
        pass
    if time.time() - t0 > TIMEOUT:
        raise TimeoutError(f"sglang не поднялся за {TIMEOUT}s — см. {LOG_PATH}")
    time.sleep(3)

In [ ]:
# Дымовой тест прямо по HTTP — убедиться, что API живой ещё до сборки пайплайна
from openai import OpenAI
_client = OpenAI(base_url=BASE_URL, api_key="EMPTY", timeout=120)
_r = _client.chat.completions.create(
    model=GEN_MODEL,
    messages=[{"role": "user", "content": "Скажи одно слово: 'привет'."}],
    temperature=0.0, max_tokens=8,
)
print("API OK →", _r.choices[0].message.content)

## 8. Модули пайплайна

vLLM в процесс больше не грузим — `Generator(openai_api)` это HTTP-клиент, поэтому
сборка пайплайна теперь почти мгновенная.

In [ ]:
import time, gc, torch
import polars as pl
from transformers import AutoTokenizer
from rag.chunking import load_chunks
from rag.retrieval.dense import DenseRetriever
from rag.retrieval.bm25 import BM25Retriever
from rag.rerank import Reranker
from rag.generation import Generator
from rag.pipeline import Pipeline
from rag.length import trim_answer
from rag.submission import write_submission
from rag.types import Answer

chunks = load_chunks(cfg.resolve(cfg.paths.chunks_parquet))
dense = DenseRetriever(cfg.embedder); dense.load(cfg.resolve(cfg.paths.faiss_index))
bm25 = BM25Retriever(); bm25.load(cfg.resolve(cfg.paths.bm25_pickle))
reranker = Reranker(cfg.reranker)

generator = Generator(cfg.generator, seed=cfg.seed)
tokenizer = AutoTokenizer.from_pretrained(cfg.chunking.tokenizer_model)
pipeline = Pipeline(cfg, chunks, dense, bm25, reranker, generator, tokenizer)

gc.collect(); torch.cuda.empty_cache()
print(f"pipeline ready: {chunks.height} chunks; свободно на GPU:",
      round(torch.cuda.mem_get_info()[0]/1e9, 1), "ГБ")

## 9. Дымовой тест пайплайна

In [ ]:
ans = pipeline.answer("debug", "Как открыть карту Альфа-Банка?")
print(f"answer ({len(ans.answer)} chars):\n{ans.answer}")

## 10. Полный прогон — батчами + чекпоинт

Retrieval/rerank остаются в этом процессе, генерация уходит на sglang по HTTP.
Внутри батча запросы летят параллельно (см. `CONCURRENCY` / `max_concurrency`).

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

CKPT = Path(WORKDIR) / "submissions" / "checkpoint.csv"
CKPT.parent.mkdir(parents=True, exist_ok=True)
no_data = cfg.length.no_data_phrase

questions = pl.read_csv(cfg.resolve(cfg.paths.questions_csv), infer_schema_length=0)
if N_QUESTIONS is not None:
    questions = questions.head(N_QUESTIONS)

done = set()
if CKPT.exists():
    done = set(pl.read_csv(CKPT, infer_schema_length=0)["q_id"].to_list())
rows = [r for r in questions.iter_rows(named=True) if str(r["q_id"]) not in done]
print(f"всего {questions.height}, сделано {len(done)}, осталось {len(rows)}")

def _append_ckpt(recs):
    df = pl.DataFrame({"q_id": [r[0] for r in recs], "answer": [r[1] for r in recs]})
    if CKPT.exists():
        with open(CKPT, "ab") as f:
            df.write_csv(f, include_header=False)
    else:
        df.write_csv(CKPT)

t0 = time.time()
for i in tqdm(range(0, len(rows), BATCH)):
    batch = rows[i:i + BATCH]
    meta = [(str(r["q_id"]), pipeline.retrieve_and_ground(str(r["q_id"]), r["query"])) for r in batch]
    to_gen = [ctx for _, ctx in meta if ctx is not None]
    gen = iter(generator.generate_batch(to_gen)) if to_gen else iter(())
    recs = []
    for qid, ctx in meta:
        if ctx is None:
            recs.append((qid, no_data))
        else:
            raw = next(gen).strip() or no_data
            recs.append((qid, trim_answer(raw, cfg.length.answer_max_chars)))
    _append_ckpt(recs)

dt = time.time() - t0
print(f"готово: {len(rows)} вопросов за {dt:.0f}s ({dt/max(len(rows),1):.2f}s/q)")

## 11. Submission + статистика

In [ ]:
ck = pl.read_csv(CKPT, infer_schema_length=0)
order = pl.read_csv(cfg.resolve(cfg.paths.questions_csv), infer_schema_length=0).select("q_id")
ck = order.join(ck, on="q_id", how="left")
answers = [Answer(q_id=str(r["q_id"]), answer=(r["answer"] or no_data)) for r in ck.iter_rows(named=True)]

OUT = cfg.resolve(cfg.paths.submission_csv)
write_submission(answers, sample_path=cfg.resolve(cfg.paths.sample_submission_csv), out_path=OUT)

out_df = pl.read_csv(OUT)
tcol = "answer_new" if "answer_new" in out_df.columns else "answer"
lens = out_df.with_columns(pl.col(tcol).str.len_chars().alias("L"))["L"]
nd = out_df.filter(pl.col(tcol) == no_data).height
print(f"rows={out_df.height}  chars: mean={lens.mean():.0f} p50={lens.quantile(0.5):.0f} p95={lens.quantile(0.95):.0f} max={lens.max()}")
print(f"no-data: {nd}/{out_df.height} ({100*nd/out_df.height:.1f}%)")

In [ ]:
from google.colab import files
files.download(str(OUT))   # скачать submission.csv локально

## 12. Остановить sglang

Аккуратно гасим фоновый процесс, чтобы освободить GPU.

In [ ]:
import signal, time
try:
    sglang_proc
except NameError:
    print("sglang не запускался в этой сессии")
else:
    if sglang_proc.poll() is None:
        sglang_proc.send_signal(signal.SIGINT)
        try:
            sglang_proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            sglang_proc.kill()
            sglang_proc.wait()
    print("sglang остановлен, exit:", sglang_proc.returncode)

## Перед отправкой (лимит 3 загрузки/день)

- [ ] Полный `questions.csv` прогнан без падений/пустых ответов.
- [ ] Длины адекватны, хвостов ≥3× нет (статистика выше).
- [ ] no-data ответы разумны (не на очевидных вопросах).
- [ ] CSV совпал с `sample_submission.csv` (гарантирует `write_submission`).
- [ ] Залогированы: `GEN_MODEL`, seed, `config.yaml`.